# 4단계 · 네트워크 그래프 시각화

**목표:** 2단계에서 확보한 연관 네트워크를 **그래프로 그려** 구조를 눈으로 파악합니다.

## 읽는 법
- **노드** = 아이템, **엣지** = 연관 관계
- **색** = 차수(🟠 시드 / 🔵 1차수 / 🟢 2차수)
- 시드에서 뻗어나가는 이웃, 이웃끼리의 연결(군집)이 보입니다.

`networkx`로 그래프를 만들고 `matplotlib`로 그립니다.

> **1단계에서 확정한 시드를 사용합니다.**
> 데모 시드: **Mathematical finance** (itemId `33589680`, 카테고리 *인공지능*, 지표 *기술집약도*)

In [ ]:
# --- APOLLO API 호출 헬퍼 (모든 노트북 공통) ---
import requests, urllib3
import pandas as pd
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

BASE_URL = "https://apollo.kisti.re.kr/service-test"   # 테스트 서버

def call_api(method, path, params=None, payload=None):
    """APOLLO 호출 후 {meta, data} 본문을 반환. (테스트 서버라 verify=False)"""
    r = requests.request(method, BASE_URL + path, params=params, json=payload,
                         headers={"Content-Type": "application/json"},
                         verify=False, timeout=120)
    body = r.json()
    if not (isinstance(body, dict) and "data" in body):
        raise RuntimeError(f"API 오류 (HTTP {r.status_code}): {body}")
    return body

print("준비 완료 · BASE_URL =", BASE_URL)

In [ ]:
# 1단계에서 확정한 시드 (다른 데모 사례로 바꾸려면 값만 교체)
SEED_ID = 33589680          # Mathematical finance
SEED_NAME = "Mathematical finance"
CATEGORY = "인공지능"
DEGREE = 3                  # APOLLO degree=3  →  사용자 체감 '2차수'(이웃의 이웃)

### 네트워크 데이터 가져오기 (2단계와 동일 호출)

In [ ]:
res = call_api("GET", f"/open/api/v1/items/{SEED_ID}/network",
               params={"degree": DEGREE, "indicator": "true"})
nodes = res["data"].get("nodes", [])
edges = res["data"].get("edges", [])
print("노드", len(nodes), "· 엣지", len(edges))

### 그래프 그리기

In [ ]:
import networkx as nx
import matplotlib.pyplot as plt

plt.rcParams["font.family"] = "Malgun Gothic"   # Windows 한글 폰트
plt.rcParams["axes.unicode_minus"] = False

# label(이름) 기준 그래프 구성
G = nx.Graph()
level_of, is_seed = {}, {}
for n in nodes:
    lab = n.get("label")
    G.add_node(lab)
    level_of[lab] = n.get("level", 1)
    is_seed[lab] = (str(n.get("id")) == str(SEED_ID)) or (lab == SEED_NAME)
for e in edges:
    if e.get("from") and e.get("to"):
        G.add_edge(e["from"], e["to"])

# 차수별 색
color_map = {0: "#ff7043", 1: "#42a5f5", 2: "#66bb6a"}
colors = [color_map.get(0 if is_seed[n] else level_of.get(n, 1), "#b0bec5") for n in G.nodes()]
sizes  = [420 if is_seed[n] else 120 for n in G.nodes()]

pos = nx.spring_layout(G, seed=42, k=1.5 / (G.number_of_nodes() ** 0.5))
plt.figure(figsize=(12, 9))
nx.draw_networkx_edges(G, pos, alpha=0.25, width=0.6)
nx.draw_networkx_nodes(G, pos, node_color=colors, node_size=sizes, linewidths=0.5, edgecolors="white")
# 시드 + 1차수만 라벨 표시(너무 빽빽해지지 않게)
labels = {n: n for n in G.nodes() if is_seed[n] or level_of.get(n) == 1}
nx.draw_networkx_labels(G, pos, labels=labels, font_size=8, font_family="Malgun Gothic")
plt.title(f"{SEED_NAME} 연관 네트워크 (degree={DEGREE})", fontsize=13)
plt.axis("off"); plt.tight_layout(); plt.show()

**해석 포인트**
- 시드(주황)를 중심으로 1차수(파랑) 이웃이 방사형으로, 그 바깥에 2차수(초록)가 붙습니다.
- 2차수끼리도 엣지가 있으면 **주제 군집**을 형성 — 유망 세부 영역을 시사합니다.